# Evaluación del Modelo YOLOv11 - Detección de Logos/Marcas

Este notebook permite evaluar el modelo YOLOv11 entrenado con diferentes imágenes de prueba.

## Objetivo
Probar el modelo de detección de logos en imágenes para verificar:
- Precisión de las detecciones
- Confianza de las predicciones
- Capacidad de detectar múltiples marcas

## 1. Importar Librerías

In [ ]:
from ultralytics import YOLO
import cv2
import glob
import os
from pathlib import Path
from IPython.display import Image, display
import matplotlib.pyplot as plt
import numpy as np

## 2. Cargar el Modelo Entrenado

In [ ]:
# Cargar modelo entrenado
model_path = '../models/models_org/weights/best.pt'
model = YOLO(model_path)

# Mostrar información del modelo
print(f"✅ Modelo cargado: {model_path}")
print(f"📊 Número de clases: {len(model.names)}")
print(f"🏷️  Clases detectables:")
for idx, name in model.names.items():
    print(f"   {idx}: {name}")

## 3. Probar con una Imagen Individual

In [ ]:
# Ruta a una imagen de prueba (cambia esta ruta según tus imágenes)
test_image = 'test_images/adidas_store.jpg'

# Hacer predicción
results = model(test_image, conf=0.25)

# Mostrar resultados
print(f"🖼️  Evaluando: {test_image}\n")

for r in results:
    if len(r.boxes) > 0:
        print(f"✅ Detectados {len(r.boxes)} logos:\n")
        for box in r.boxes:
            brand = model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            bbox = box.xyxy[0].tolist()
            print(f"   📌 {brand}")
            print(f"      Confianza: {conf:.2%}")
            print(f"      Bounding box: {bbox}\n")
    else:
        print("⚠️ No se detectaron logos en esta imagen")

## 4. Evaluar Todas las Imágenes del Directorio

In [ ]:
# Buscar todas las imágenes en el directorio de prueba
test_images_dir = 'test_images'
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']

test_images = []
for ext in image_extensions:
    test_images.extend(glob.glob(f'{test_images_dir}/{ext}'))
    test_images.extend(glob.glob(f'{test_images_dir}/{ext.upper()}'))

print(f"📁 Encontradas {len(test_images)} imágenes en {test_images_dir}/\n")
print("="*60)

# Estadísticas generales
total_detections = 0
brands_detected = set()
confidences = []

# Evaluar cada imagen
for img_path in test_images:
    print(f"\n🖼️  {os.path.basename(img_path)}")
    
    results = model(img_path, conf=0.25, verbose=False)
    
    for r in results:
        if len(r.boxes) > 0:
            for box in r.boxes:
                brand = model.names[int(box.cls[0])]
                conf = float(box.conf[0])
                
                brands_detected.add(brand)
                confidences.append(conf)
                total_detections += 1
                
                print(f"   ✅ {brand}: {conf:.2%}")
        else:
            print(f"   ⚠️  Sin detecciones")

print("\n" + "="*60)
print("📊 RESUMEN")
print("="*60)
print(f"Total de imágenes evaluadas: {len(test_images)}")
print(f"Total de detecciones: {total_detections}")
print(f"Marcas únicas detectadas: {len(brands_detected)}")
if brands_detected:
    print(f"   {', '.join(brands_detected)}")
if confidences:
    print(f"Confianza promedio: {np.mean(confidences):.2%}")
    print(f"Confianza mínima: {np.min(confidences):.2%}")
    print(f"Confianza máxima: {np.max(confidences):.2%}")
print("="*60)

## 5. Visualizar Resultados

Guardar imágenes con las detecciones visualizadas

In [ ]:
# Procesar y guardar imágenes con detecciones
output_dir = 'results/detections'
os.makedirs(output_dir, exist_ok=True)

print(f"💾 Guardando imágenes con detecciones en: {output_dir}/\n")

for img_path in test_images[:3]:  # Procesar las primeras 3 imágenes como ejemplo
    print(f"Procesando: {os.path.basename(img_path)}")
    
    # Hacer predicción y guardar resultado
    results = model(img_path, conf=0.25, save=True, project=output_dir)
    
print(f"\n✅ Imágenes guardadas en: {output_dir}/")
print("   Puedes revisar las imágenes con las detecciones visualizadas")